# B.8 依能力與成本，何時值得藉助工具？


直接答可能較快，工具能給可核對結果卻多等待。是否值得用，先量同題型兩條路的完整成功率，再設定錯答與額外成本，不能把下一token的高機率當整題自信。

紙上假設直接成功0.90，工具完整往返成功0.99，錯答損失10點，工具額外成本0.02點。平均損失是 `(1-成功率)×錯答代價+額外成本`。點數是人為尺度，不是秒或美元。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
direct_accuracy = 0.90
tool_accuracy = 0.99
wrong_cost = 10.0
tool_extra_cost = 0.02
direct_loss = (1 - direct_accuracy) * wrong_cost
tool_loss = (1 - tool_accuracy) * wrong_cost + tool_extra_cost
print("直接回答平均損失", round(direct_loss, 3))
print("工具流程平均損失", round(tool_loss, 3))
print("本例選擇", "TOOL" if tool_loss < direct_loss else "DIRECT")

直接回答平均損失 1.0
工具流程平均損失 0.12
本例選擇 TOOL


直接損失1.0，工具0.12，所以外層規則選TOOL。若直接成功率改0.999，損失0.01便選DIRECT。這些數值是假設，程式沒有估信心、計時或訓練選卡器。

工具成功率包括工具與參數正確、真正執行、回填後回答正確，不能只用函數計算無誤替整條路打分。前面工具算81而模型寫8的錯例正說明這點。

要變成實際策略，先用獨立驗證題量兩路品質和等待，固定錯答代價與工具額外成本，再依主文規則：工具平均損失比直接回答低，才選TOOL。最後在未參與策略選擇的題目檢查。改問法、數字範圍或工具條件後重新檢查適用性；不能看最後哪題錯才替那題設特別門檻。這是固定工作策略的延伸，而不是已經訓練出來的逐題信心能力。
